# Train SegFormer on MARIDA
This notebook demonstrates how to load MARIDA satellite patches (11 band Sentinel-2 images) and fine-tune a SegFormer semantic segmentation model.


In [1]:
import os
import rasterio
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import SegformerForSemanticSegmentation, TrainingArguments, Trainer
import evaluate


c:\Users\navee\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class MARIDADataset(Dataset):
    def __init__(self, split_file, patches_dir):
        with open(split_file, 'r') as f:
            self.patch_names = f.read().splitlines()
        self.patches_dir = patches_dir

    def __len__(self):
        return len(self.patch_names)

    def __getitem__(self, idx):
        patch_name = self.patch_names[idx]
        prefix = "S2_" + "_".join(patch_name.split("_")[:-1])
        full_name = "S2_" + patch_name
        patch_path = os.path.join(self.patches_dir, prefix, full_name + ".tif")
        cl_path = os.path.join(self.patches_dir, prefix, full_name + "_cl.tif")
        
        with rasterio.open(patch_path) as src:
            image = src.read().astype(np.float32)
            # Perform basic min-max normalization
            # Sentinel-2 data typically ranges up to ~10000 
            image = np.nan_to_num(image)
            image = np.clip(image / 10000.0, 0, 1)

        with rasterio.open(cl_path) as src:
            # Classes are 0 (unclassified) + 1 to 15
            mask = src.read(1).astype(np.int64)
            
        return {"pixel_values": torch.tensor(image), "labels": torch.tensor(mask)}


In [3]:
dataset_path = 'dataset/MARIDA'
train_split = os.path.join(dataset_path, 'splits', 'train_X.txt')
val_split = os.path.join(dataset_path, 'splits', 'val_X.txt')
patches_dir = os.path.join(dataset_path, 'patches')

train_dataset = MARIDADataset(train_split, patches_dir)
val_dataset = MARIDADataset(val_split, patches_dir)

print(f"Training samples: len(train_dataset)")
print(f"Validation samples: len(val_dataset)")
sample = train_dataset[0]
print(f"Inputs shape: {sample['pixel_values'].shape}, Mask shape: {sample['labels'].shape}")


Training samples: len(train_dataset)
Validation samples: len(val_dataset)
Inputs shape: torch.Size([11, 256, 256]), Mask shape: torch.Size([256, 256])


In [4]:
# 16 classes including unclassified (0)
num_labels = 16
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0", 
    num_labels=num_labels, 
    ignore_mismatched_sizes=True
)

# Replace the first convolutional layer to accept 11 channels instead of 3
original_conv = model.segformer.encoder.patch_embeddings[0].proj
new_conv = torch.nn.Conv2d(11, 32, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))

# Randomly initialized new weights, but we can copy 3 channels for faster convergence (optional)
with torch.no_grad():
    new_conv.weight[:, :3, :, :] = original_conv.weight

model.segformer.encoder.patch_embeddings[0].proj = new_conv

print("Model modified for 11 channel input.")


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model modified for 11 channel input.


In [5]:
# Define a simple evaluation metric using mean IoU
metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        # upscale logits to the label sizes
        logits_tensor = torch.nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)
        
        pred_labels = logits_tensor.detach().cpu().numpy()
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=num_labels,
            ignore_index=255,
            reduce_labels=False,
        )
        return {"mean_iou": metrics["mean_iou"], "mean_accuracy": metrics["mean_accuracy"]}


In [6]:
training_args = TrainingArguments(
    output_dir="./segformer-marida-output",
    learning_rate=6e-5,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_total_limit=3,
    evaluation_strategy="steps",
    save_strategy="steps",
    save_steps=100,
    eval_steps=100,
    logging_steps=10,
    eval_accumulation_steps=5,
    remove_unused_columns=False,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


c:\Users\navee\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [7]:
torch.cuda.is_available()

True

In [8]:
# Uncomment to start training
trainer.train()


  2%|▏         | 10/435 [00:09<05:52,  1.20it/s]

{'loss': 2.7648, 'grad_norm': 6.6078057289123535, 'learning_rate': 5.862068965517242e-05, 'epoch': 0.11}


  5%|▍         | 20/435 [00:18<05:41,  1.21it/s]

{'loss': 2.5297, 'grad_norm': 5.482105255126953, 'learning_rate': 5.724137931034483e-05, 'epoch': 0.23}


  7%|▋         | 30/435 [00:26<05:39,  1.19it/s]

...

 99%|█████████▉| 430/435 [09:30<00:04,  1.10it/s]

{'loss': 0.6018, 'grad_norm': 2.709095001220703, 'learning_rate': 6.896551724137931e-07, 'epoch': 4.94}


100%|██████████| 435/435 [09:34<00:00,  1.32s/it]

{'train_runtime': 574.5088, 'train_samples_per_second': 6.04, 'train_steps_per_second': 0.757, 'train_loss': 1.1159646154820233, 'epoch': 5.0}


TrainOutput(global_step=435, training_loss=1.1159646154820233, metrics={'train_runtime': 574.5088, 'train_samples_per_second': 6.04, 'train_steps_per_second': 0.757, 'total_flos': 5.5995781349376e+16, 'train_loss': 1.1159646154820233, 'epoch': 5.0})